In [1]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV    = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY  = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')   # (T, N, 2) u,v

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, DROPOUT = 32, 1, 4, 0.2
GCN_HOPS       = 2
LOSS_VOL_ALPHA = 1.0
SEEDS          = [42, 7, 123, 2024, 999, 2025, 77]

# --- v3b: wind-aware graph, done properly ---
WIND_RADIUS_KM   = 35.0      # was 12 — reach real transport corridors
ADVECTION_TAU    = 4.0       # hours — advection-time edge-weight decay
WIND_IN_FEATURES = False     # False = wind reaches the model ONLY via the graph (clean RG6 test)
V3_TAG           = 'watgcn_v3b'

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'radius {WIND_RADIUS_KM} km | tau {ADVECTION_TAU} h | wind in features: {WIND_IN_FEATURES}')

Device: cuda
GPU   : NVIDIA GeForce RTX 3050 6GB Laptop GPU
radius 35.0 km | tau 4.0 h | wind in features: False


In [2]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arctan2(np.sqrt(a), np.sqrt(1-a))

dist_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            dist_matrix[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])
within_threshold = (dist_matrix > 0) & (dist_matrix <= WIND_RADIUS_KM)
dir_u = np.zeros((N, N)); dir_v = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            de = (lons[j]-lons[i]) * np.cos(np.radians(lats[i]))
            dn = lats[j]-lats[i]; nr = np.hypot(de, dn)
            if nr > 0:
                dir_u[i, j], dir_v[i, j] = de/nr, dn/nr

print(f'N={N}  T={T}  ERA5 mean speed {era5_spd.mean():.2f} m/s')
print(f'pairs within {WIND_RADIUS_KM:.0f} km: {int(within_threshold.sum())}  (was 508 at 12 km)')

N=37  T=19921  ERA5 mean speed 2.13 m/s
pairs within 35 km: 1314  (was 508 at 12 km)


In [3]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

REST = ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
        'city_mean_pm','city_max_pm','pm_minus_city',
        'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
Xr = np.zeros((T, N, len(BASE_FEATS + REST)), dtype=np.float32)
for fi, col in enumerate(BASE_FEATS + REST):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xr[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

if WIND_IN_FEATURES:
    FEATURE_COLS = BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + REST
    X_raw = np.concatenate([Xr[:, :, :8], era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None], Xr[:, :, 8:]], axis=2).astype(np.float32)
else:
    FEATURE_COLS = BASE_FEATS + REST
    X_raw = Xr.astype(np.float32)                    # 23 features, NO wind
F = len(FEATURE_COLS)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
FUT_IDX   = TIME_IDX + ([FEATURE_COLS.index(c) for c in ['u_wind','v_wind','wspd']] if WIND_IN_FEATURES else [])

print(f'F = {F}   WIND_IN_FEATURES = {WIND_IN_FEATURES}   n_fut = {len(FUT_IDX)}')
print(FEATURE_COLS)
print(f'X_raw {X_raw.shape}   NaNs {int(np.isnan(X_raw).sum())}')

F = 23   WIND_IN_FEATURES = False   n_fut = 7
['pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'at', 'pm_roll6_mean', 'pm_roll6_std', 'pm_roll24_mean', 'pm_roll24_std', 'pm_delta3', 'city_mean_pm', 'city_max_pm', 'pm_minus_city', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend']
X_raw (19921, 37, 23)   NaNs 0


In [4]:
def compute_At_adv(u, v):
    align = u[:, None]*dir_u + v[:, None]*dir_v                    # wind speed component i->j, m/s
    downwind = within_threshold & (align > 0.0)
    ttime = dist_matrix / (np.maximum(align, 0.05) * 3.6)         # hours (km / km-h^-1)
    w = np.where(downwind, np.exp(-ttime / ADVECTION_TAU), 0.0).astype(np.float32)
    rs = w.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return w / rs

t0 = time.time()
dyn_adj = np.zeros((T, N, N), dtype=np.float32)
for t in range(T):
    dyn_adj[t] = compute_At_adv(era5[t, :, 0], era5[t, :, 1])
np.save(os.path.join(V2_DIR, f'dynamic_adj_{V3_TAG}.npy'), dyn_adj)
print(f'advection A(t): {dyn_adj.shape} in {time.time()-t0:.0f}s')
print(f'mean edges/hour: {np.mean((dyn_adj>0).sum((1,2))):.1f}   '
      f'mean edge weight: {dyn_adj[dyn_adj>0].mean():.3f}   '
      f'reach: median downwind hop {np.median(dist_matrix[within_threshold]):.0f} km')

advection A(t): (19921, 37, 37) in 1s
mean edges/hour: 666.9   mean edge weight: 0.054   reach: median downwind hop 14 km


In [5]:
train_end_idx = int(T * 0.70); val_end_idx = int(T * 0.85)

def split_starts(lo, hi):
    return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
train_starts = split_starts(0, train_end_idx)
val_starts   = split_starts(train_end_idx, val_end_idx)
test_starts  = split_starts(val_end_idx, T)
print(f'windows -- train {len(train_starts):,} | val {len(val_starts):,} | test {len(test_starts):,}')

tr_mask = np.zeros(T, bool); tr_mask[:train_end_idx] = True
feat_scaler = StandardScaler().fit(X_raw[tr_mask].reshape(-1, F))
Xs_full = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(T, N, F).astype(np.float32)

pm_train   = Y_raw[tr_mask].reshape(-1, 1)
log_scaler = StandardScaler().fit(np.log1p(np.clip(pm_train, 0, None)))
def to_log(a):
    return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
def from_log(a):
    return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
Yl_full = to_log(Y_raw)

_std = Xs_full[tr_mask].reshape(-1, F).std(0)
print(f'per-feature std after scaling: min {_std.min():.2f}  max {_std.max():.2f}')
del X_raw
# HOW TO CHECK: window counts 13873 / 2917 / 2918. std ~1.

windows -- train 13,873 | val 2,917 | test 2,918
per-feature std after scaling: min 1.00  max 1.00


In [6]:
class EcoDS(Dataset):
    def __init__(self, starts, weights=None):
        self.s = np.asarray(starts, np.int64)
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)
    def __len__(self):
        return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(Xs_full[g:i1].copy()),                       # (48,N,26)
                torch.from_numpy(dyn_adj[g:i1].copy()),                       # (48,N,N) encoder A(t)
                torch.from_numpy(dyn_adj[i1:i2].copy()),                      # (24,N,N) forecast A_fc
                torch.from_numpy(np.ascontiguousarray(Xs_full[i1:i2][:, :, FUT_IDX])),  # (24,N,10)
                torch.from_numpy(Yl_full[i1-1].copy()),                       # (N,) anchor (log)
                torch.from_numpy(Yl_full[i1:i2].copy()),                      # (24,N) target (log)
                torch.tensor(self.w[k], dtype=torch.float32))

def window_swing(starts):
    return np.array([(Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].max(0) -
                      Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].min(0)).mean()
                     for g in starts], dtype=np.float32)
tr_sw = window_swing(train_starts); med = float(np.median(tr_sw))
tr_w = np.clip(1.0 + LOSS_VOL_ALPHA * (tr_sw / (med + 1e-6)), 0.5, 4.0).astype(np.float32)
tr_w *= len(tr_w) / tr_w.sum()

tr_ld = DataLoader(EcoDS(train_starts, tr_w), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
va_ld = DataLoader(EcoDS(val_starts),         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
te_ld = DataLoader(EcoDS(test_starts),        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'batches -- train {len(tr_ld)} | val {len(va_ld)} | test {len(te_ld)}   weights {tr_w.min():.2f}-{tr_w.max():.2f}')

batches -- train 868 | val 183 | test 183   weights 0.56-1.87


In [7]:
class WATGCNv3(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_fut=10, gcn_hops=2):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))   # 7 cyclical + 3 forecast wind
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH = H, forecast_horizon

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        A = self._n(At + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(torch.matmul(A, h)))) + h                    # 2-hop residual GCN
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)         # (B,n,H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)     # forecast-graph propagation
        fut_emb = self.fut_enc(Fut)                                           # (B,24,n,H)  per-station
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, fut_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

_m = WATGCNv3(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(FUT_IDX), GCN_HOPS)
print(f'v3 parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m

v3 parameters: 18,105


In [8]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def score(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())), preds=pr, actual=ar)

def train_model(seed, max_epochs=60, patience=12, lr=1e-3, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    model = WATGCNv3(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(FUT_IDX), GCN_HOPS).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(tr_ld) * max_epochs; warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s/warm if s < warm else 0.5*(1 + np.cos(np.pi*(s-warm)/max(1, steps-warm))))
    best, best_state, ctr = 1e9, None, 0
    hist = {'train': [], 'val': []}
    for ep in range(1, max_epochs+1):
        t0 = time.time()
        tr = run_epoch(model, tr_ld, opt, sched)
        vl = run_epoch(model, va_ld)
        hist['train'].append(tr); hist['val'].append(vl)
        if vl < best - 1e-5:
            best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
            tag = 'best'
        else:
            ctr += 1; tag = f'{ctr}/{patience}'
        if verbose and (ep % 3 == 0 or tag == 'best' or ctr >= patience):
            print(f'  ep {ep:>2} | train {tr:.5f} | val {vl:.5f} | {time.time()-t0:>4.0f}s | {tag}')
        if ctr >= patience:
            break
    model.load_state_dict(best_state)
    return model, best, hist

print('ready')

ready


In [9]:
_m, _b, _h = train_model(42, max_epochs=3, patience=99)
_p, _a = predict(_m, va_ld)
print('val (3 ep) ->', {k: round(v, 2) for k, v in score(_p, _a).items() if k in ('rmse','mae','r2')})
del _m; torch.cuda.empty_cache()
# HOW TO CHECK: val loss falls each epoch; RMSE ~55-75 after 3 epochs. NaN/flat -> stop.

  ep  1 | train 0.16479 | val 0.13800 |   20s | best
  ep  3 | train 0.13019 | val 0.13732 |   19s | best
val (3 ep) -> {'rmse': 55.23, 'mae': 31.58, 'r2': 0.74}


In [10]:
models, val_scores, hists, val_preds = [], [], [], []
for sd in SEEDS:
    print(f'\n=== seed {sd} ===')
    m, b, h = train_model(sd)
    p, a = predict(m, va_ld)
    models.append(m); val_scores.append(b); hists.append(h); val_preds.append(p)
    print(f'  seed {sd}: val RMSE {score(p, a)["rmse"]:.2f}')
val_actual = a
print(f'\nper-seed val RMSE: {[round(score(p, val_actual)["rmse"], 2) for p in val_preds]}')


=== seed 42 ===
  ep  1 | train 0.20654 | val 0.15418 |   19s | best
  ep  2 | train 0.15092 | val 0.14608 |   19s | best
  ep  3 | train 0.13784 | val 0.13842 |   19s | best
  ep  4 | train 0.13216 | val 0.13452 |   19s | best
  ep  6 | train 0.12259 | val 0.13953 |   19s | 2/12
  ep  9 | train 0.11093 | val 0.14629 |   19s | 5/12
  ep 12 | train 0.10232 | val 0.15054 |   19s | 8/12
  ep 15 | train 0.09663 | val 0.16585 |   19s | 11/12
  ep 16 | train 0.09505 | val 0.15767 |   19s | 12/12
  seed 42: val RMSE 56.68

=== seed 7 ===
  ep  1 | train 0.20579 | val 0.15243 |   19s | best
  ep  2 | train 0.15016 | val 0.13727 |   19s | best
  ep  3 | train 0.13804 | val 0.14860 |   19s | 1/12
  ep  6 | train 0.12050 | val 0.14221 |   19s | 4/12
  ep  9 | train 0.10791 | val 0.16086 |   19s | 7/12
  ep 12 | train 0.10018 | val 0.14669 |   29s | 10/12
  ep 14 | train 0.09651 | val 0.14887 |   29s | 12/12
  seed 7: val RMSE 56.40

=== seed 123 ===
  ep  1 | train 0.21138 | val 0.16342 |   29s 

In [11]:
val_ens = score(np.mean(val_preds, axis=0), val_actual)
test_seeds = [predict(m, te_ld)[0] for m in models]
_, test_actual = predict(models[0], te_ld)
test_ens = score(np.mean(test_seeds, axis=0), test_actual)

v1 = json.load(open(os.path.join(RESULTS_DIR, 'final_metrics.json')))
v2 = json.load(open(os.path.join(RESULTS_DIR, 'v2_metrics.json')))

print(f"{'':12}{'v1':>9}{'v2':>9}{'v3 val':>9}{'v3 test':>9}")
print(f"{'RMSE':12}{v1['rmse']:>9.2f}{v2['rmse']:>9.2f}{val_ens['rmse']:>9.2f}{test_ens['rmse']:>9.2f}")
print(f"{'MAE':12}{v1['mae']:>9.2f}{v2['mae']:>9.2f}{val_ens['mae']:>9.2f}{test_ens['mae']:>9.2f}")
print(f"{'R2':12}{v1['r2']:>9.3f}{v2['r2']:>9.3f}{val_ens['r2']:>9.3f}{test_ens['r2']:>9.3f}")

pr, ar = test_ens['preds'], test_ens['actual']
h_rmse = [float(np.sqrt(mean_squared_error(ar[:, h, :].ravel(), pr[:, h, :].ravel()))) for h in range(HORIZON)]
print('\nRMSE by horizon (v2 -> v3):')
for h in [0, 5, 11, 17, 23]:
    print(f"  +{h+1:>2}h : {v2['rmse_by_horizon'][f'+{h+1}h']:>6.1f}  ->  {h_rmse[h]:>6.1f}")

swing = ar.max(1) - ar.min(1)
q33, q66 = np.percentile(swing, [33, 66])
reg = {'Stable': swing <= q33, 'Moderate': (swing > q33) & (swing <= q66), 'Volatile': swing > q66}
print(f"\n{'regime':10}{'v2':>9}{'v3':>9}")
reg_rmse = {}
for k, msk in reg.items():
    a_, p_ = ar.transpose(0, 2, 1)[msk], pr.transpose(0, 2, 1)[msk]
    reg_rmse[k] = float(np.sqrt(((a_ - p_)**2).mean()))
    print(f"{k:10}{v2['rmse_by_regime'][k]:>9.2f}{reg_rmse[k]:>9.2f}")

                   v1       v2   v3 val  v3 test
RMSE            64.59    60.92    55.51    58.79
MAE             41.50    39.69    29.97    38.10
R2              0.589    0.634    0.735    0.659

RMSE by horizon (v2 -> v3):
  + 1h :   41.4  ->    41.2
  + 6h :   56.7  ->    56.0
  +12h :   63.0  ->    60.7
  +18h :   66.0  ->    62.9
  +24h :   66.5  ->    63.1

regime           v2       v3
Stable        29.89    27.81
Moderate      42.95    41.00
Volatile      90.87    88.22


In [12]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].plot(range(1, len(hists[0]['train'])+1), hists[0]['train'], marker='o', ms=3, label='train')
ax[0].plot(range(1, len(hists[0]['val'])+1),   hists[0]['val'],   marker='s', ms=3, label='val')
ax[0].set_title(f'seed {SEEDS[0]} training'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(range(1, 25), [v2['rmse_by_horizon'][f'+{h+1}h'] for h in range(24)], marker='o', ms=3, label='v2')
ax[1].plot(range(1, 25), h_rmse, marker='s', ms=3, label='v3 (ERA5 wind)')
ax[1].set_title('RMSE by horizon'); ax[1].set_xlabel('hours ahead'); ax[1].set_ylabel('RMSE ug/m3')
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'v3_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_11276\2615813367.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
for i, m in enumerate(models):
    torch.save(m.state_dict(), os.path.join(MODEL_DIR, f'{V3_TAG}_seed{SEEDS[i]}.pt'))
json.dump({'in_features': F, 'hidden_dim': HIDDEN_DIM, 'gru_layers': GRU_LAYERS, 'attn_heads': ATTN_HEADS,
           'forecast_horizon': HORIZON, 'n_stations': N, 'dropout': DROPOUT, 'input_window': INPUT_WINDOW,
           'gcn_hops': GCN_HOPS, 'n_fut': len(FUT_IDX), 'feature_cols': FEATURE_COLS, 'fut_cols': TIME_COLS + ['u_wind','v_wind','wspd'],
           'wind_source': 'ERA5 reanalysis (Open-Meteo)', 'forecast_wind': 'oracle (ERA5 at horizon hours)',
           'log_target': True, 'vol_loss_alpha': LOSS_VOL_ALPHA, 'seeds': SEEDS},
          open(os.path.join(MODEL_DIR, 'v3_config.json'), 'w'), indent=2)
pickle.dump(feat_scaler, open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'wb'))
pickle.dump(log_scaler,  open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'wb'))
json.dump({'model': 'WA-TGCN v3 (ERA5 wind + oracle forecast wind, 7-seed)',
           'val_rmse': round(val_ens['rmse'], 2), 'rmse': round(test_ens['rmse'], 2),
           'mae': round(test_ens['mae'], 2), 'r2': round(test_ens['r2'], 4),
           'rmse_by_horizon': {f'+{h+1}h': round(h_rmse[h], 2) for h in range(HORIZON)},
           'rmse_by_regime': {k: round(reg_rmse[k], 2) for k in reg_rmse}},
          open(os.path.join(RESULTS_DIR, 'v3_metrics.json'), 'w'), indent=2)
print('saved: 7 v3 models, v3_config.json, v3 scalers, v3_metrics.json, dynamic_adj_om.npy')
print(f"\nv1 64.59  ->  v2 60.92  ->  v3 {test_ens['rmse']:.2f}   (R2 {test_ens['r2']:.3f})")

saved: 7 v3 models, v3_config.json, v3 scalers, v3_metrics.json, dynamic_adj_om.npy

v1 64.59  ->  v2 60.92  ->  v3 58.79   (R2 0.659)
